# Step 3 — Build the RAG Chain
Implements full retrieval-augmented generation using:
- **Retrieval:** Cortex Search Service (from Step 2)
- **Generation:** `SNOWFLAKE.CORTEX.COMPLETE()` with llama3.1-70b
- **Citation:** Every answer includes inline [Source N] references

**Constraints:** Cortex-only LLM inference. No OpenAI / Anthropic / external APIs.

In [ ]:
# STEP 3 — CELL 1
import json
import pandas as pd
from snowflake.snowpark import Session
from snowflake.cortex import Complete
from snowflake.core import Root

try:
    session = Session.builder.getOrCreate()
except Exception:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()

TARGET_DB     = "RAG_HACKATHON_DB"
TARGET_SCHEMA = "RAG_SCHEMA"
TARGET_WH     = "RAG_SEARCH_WH"   # use the warehouse you created in Step 2
SERVICE_NAME  = "ECONOMIC_SEARCH"
LLM_MODEL     = "llama3.1-70b"
TOP_K         = 5

session.sql(f"USE DATABASE {TARGET_DB}").collect()
session.sql(f"USE SCHEMA {TARGET_SCHEMA}").collect()
session.sql(f"USE WAREHOUSE {TARGET_WH}").collect()

root = Root(session)
svc = root.databases[TARGET_DB].schemas[TARGET_SCHEMA].cortex_search_services[SERVICE_NAME]

print("RAG chain ready")
print("DB:", session.get_current_database())
print("Schema:", session.get_current_schema())
print("WH:", session.get_current_warehouse())
print("Service:", SERVICE_NAME)
print("Model:", LLM_MODEL)

In [ ]:
# CELL 2 — Core RAG functions (final cleaner version)

def retrieve(question: str, k: int = TOP_K, source_filter: str = None) -> list:
    """Retrieve top-k relevant chunks from Cortex Search."""
    kwargs = dict(
        query=question,
        columns=['chunk_text', 'source', 'doc_id', 'title', 'chunk_index'],
        limit=k
    )
    if source_filter:
        kwargs['filter'] = {'@eq': {'source': source_filter}}
    resp = svc.search(**kwargs)
    return resp.model_dump().get('results', [])


def build_prompt(question: str, chunks: list) -> tuple:
    """Build grounded prompt with numbered source labels. Returns (prompt, sources_list)."""
    context_str = ''
    sources = []
    for i, chunk in enumerate(chunks):
        label = f"[Source {i+1}: {chunk.get('title', 'N/A')[:60]} | Dataset: {chunk['source']}]"
        context_str += f'\n{label}\n{chunk["chunk_text"]}\n'
        sources.append({
            'id': i + 1,
            'title': chunk.get('title', 'N/A'),
            'source': chunk['source'],
            'doc_id': chunk.get('doc_id', ''),
            'snippet': chunk['chunk_text'][:300]
        })

    prompt = f"""You are an economic and financial intelligence assistant.
Answer the user's question using ONLY the provided context below.
Include inline citations like [Source N] after each fact you use.
If the context does not contain enough information, say: 'The available data does not cover this topic.'
Do NOT make up facts. Do NOT use external knowledge beyond the context.

Context:
{context_str}

Question: {question}

Answer (with [Source N] inline citations):"""
    return prompt, sources


def generate(prompt: str) -> str:
    """Call Snowflake Cortex COMPLETE — fully within Snowflake, no external API."""
    return Complete(LLM_MODEL, prompt, session=session)


def rag_query(question: str, k: int = TOP_K, source_filter: str = None) -> dict:
    """Full RAG pipeline: retrieve → prompt → generate."""
    chunks = retrieve(question, k=k, source_filter=source_filter)
    prompt, sources = build_prompt(question, chunks)
    answer = generate(prompt)
    return {
        'question': question,
        'answer': answer,
        'sources': sources,
        'chunks': chunks
    }


print('RAG functions defined')

In [ ]:
# CELL 3 — Test with sample questions (better source targeting)

tests = [
    ("What are the current trends in loan default rates?", "banking"),
    ("What does the data show about household income distribution?", "demographics"),
    ("How do demographic factors relate to banking behavior?", None),
]

for q, src in tests:
    print("\n" + "=" * 70)
    print("Q:", q)

    result = rag_query(q, source_filter=src)

    print("\nA:")
    print(result["answer"][:800] + ("..." if len(result["answer"]) > 800 else ""))

    print("\nSources used:")
    for s in result["sources"]:
        print(f'  [{s["id"]}] {s["title"][:55]} ({s["source"]})')

In [ ]:
# CELL 4 — Save RAG results to Snowflake table (for audit / eval)
import datetime

def log_query(question, answer, sources):
    """Persist RAG Q&A to a log table inside Snowflake."""
    session.sql(f"""
    CREATE TABLE IF NOT EXISTS {TARGET_DB}.{TARGET_SCHEMA}.RAG_QUERY_LOG (
        query_id   VARCHAR DEFAULT UUID_STRING(),
        question   VARCHAR,
        answer     VARCHAR,
        sources    VARIANT,
        model      VARCHAR,
        top_k      INT,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
    )""").collect()
    sources_json = json.dumps(sources).replace("'","\\'")
    session.sql(f"""
    INSERT INTO {TARGET_DB}.{TARGET_SCHEMA}.RAG_QUERY_LOG (question, answer, sources, model, top_k)
    SELECT
      $${question}$$,
      $${answer}$$,
      PARSE_JSON($${sources_json}$$),
      '{LLM_MODEL}',
      {TOP_K}
    """).collect()

# Test logging
r = rag_query('What banking metrics are tracked in the dataset?')
log_query(r['question'], r['answer'], r['sources'])
print('Query logged to RAG_QUERY_LOG')
print(f'Answer preview: {r["answer"][:400]}')

print('\nStep 3 COMPLETE — RAG chain is operational')